In [3]:
import pandas as pd

df = pd.read_csv("dataset.csv")

print(df.shape)
print(df.head())
print(df.columns)

(975, 30)
           datetime  underlying_price  NIFTY27JAN2625200CE  \
0  07-01-2026 09:15          26111.65              0.12662   
1  07-01-2026 09:20          26141.40              0.08632   
2  07-01-2026 09:25          26139.35              0.09147   
3  07-01-2026 09:30          26128.95              0.10860   
4  07-01-2026 09:35          26131.90              0.10462   

   NIFTY27JAN2625300CE  NIFTY27JAN2625400CE  NIFTY27JAN2625500CE  \
0              0.12330              0.11741                  NaN   
1                  NaN                  NaN              0.11779   
2                  NaN              0.09514              0.09933   
3              0.10842              0.11150              0.12248   
4              0.10538              0.12459              0.12051   

   NIFTY27JAN2625600CE  NIFTY27JAN2625700CE  NIFTY27JAN2625800CE  \
0              0.11005              0.10576                  NaN   
1              0.11197              0.11028                  NaN   
2   

In [4]:
print(df.isna().sum())

datetime                 0
underlying_price         0
NIFTY27JAN2625200CE    194
NIFTY27JAN2625300CE    184
NIFTY27JAN2625400CE    202
NIFTY27JAN2625500CE    181
NIFTY27JAN2625600CE    180
NIFTY27JAN2625700CE    201
NIFTY27JAN2625800CE    210
NIFTY27JAN2625900CE    182
NIFTY27JAN2626000CE    181
NIFTY27JAN2626100CE    183
NIFTY27JAN2626200CE    209
NIFTY27JAN2626300CE    187
NIFTY27JAN2626400CE    213
NIFTY27JAN2626500CE    184
NIFTY27JAN2623800PE    203
NIFTY27JAN2623900PE    201
NIFTY27JAN2624000PE    209
NIFTY27JAN2624100PE    176
NIFTY27JAN2624200PE    217
NIFTY27JAN2624300PE    189
NIFTY27JAN2624400PE    202
NIFTY27JAN2624500PE    195
NIFTY27JAN2624600PE    198
NIFTY27JAN2624700PE    175
NIFTY27JAN2624800PE    209
NIFTY27JAN2624900PE    195
NIFTY27JAN2625000PE    195
NIFTY27JAN2625100PE    205
dtype: int64


In [5]:
print(df.isna().sum().sum())

5460


In [6]:
print(df.columns.tolist())

['datetime', 'underlying_price', 'NIFTY27JAN2625200CE', 'NIFTY27JAN2625300CE', 'NIFTY27JAN2625400CE', 'NIFTY27JAN2625500CE', 'NIFTY27JAN2625600CE', 'NIFTY27JAN2625700CE', 'NIFTY27JAN2625800CE', 'NIFTY27JAN2625900CE', 'NIFTY27JAN2626000CE', 'NIFTY27JAN2626100CE', 'NIFTY27JAN2626200CE', 'NIFTY27JAN2626300CE', 'NIFTY27JAN2626400CE', 'NIFTY27JAN2626500CE', 'NIFTY27JAN2623800PE', 'NIFTY27JAN2623900PE', 'NIFTY27JAN2624000PE', 'NIFTY27JAN2624100PE', 'NIFTY27JAN2624200PE', 'NIFTY27JAN2624300PE', 'NIFTY27JAN2624400PE', 'NIFTY27JAN2624500PE', 'NIFTY27JAN2624600PE', 'NIFTY27JAN2624700PE', 'NIFTY27JAN2624800PE', 'NIFTY27JAN2624900PE', 'NIFTY27JAN2625000PE', 'NIFTY27JAN2625100PE']


In [7]:
df['datetime'] = pd.to_datetime(df['datetime'])

print(df['datetime'].min())
print(df['datetime'].max())

ValueError: time data "13-01-2026 09:15" doesn't match format "%m-%d-%Y %H:%M". You might want to try:
    - passing `format` if your strings have a consistent format;
    - passing `format='ISO8601'` if your strings are all ISO8601 but not necessarily in exactly the same format;
    - passing `format='mixed'`, and the format will be inferred for each element individually. You might want to use `dayfirst` alongside this.

In [8]:
df['datetime'] = pd.to_datetime(
    df['datetime'],
    format='%d-%m-%Y %H:%M'
)

print(df['datetime'].min())
print(df['datetime'].max())

2026-01-07 09:15:00
2026-01-27 15:25:00


In [9]:
df = df.sort_values('datetime').reset_index(drop=True)

split_idx = int(0.8 * len(df))

train_df = df.iloc[:split_idx].copy()
val_df = df.iloc[split_idx:].copy()

print(train_df.shape)
print(val_df.shape)

(780, 30)
(195, 30)


In [10]:
iv_cols = [
    col for col in df.columns
    if col not in ['datetime', 'underlying_price']
]

print(len(iv_cols))
print(iv_cols[:5])

28
['NIFTY27JAN2625200CE', 'NIFTY27JAN2625300CE', 'NIFTY27JAN2625400CE', 'NIFTY27JAN2625500CE', 'NIFTY27JAN2625600CE']


In [11]:
val_masked = val_df.copy()

In [12]:
import numpy as np

np.random.seed(42)

hidden_positions = []

for col in iv_cols:

    valid_idx = val_masked[
        val_masked[col].notna()
    ].index

    hide_count = int(0.1 * len(valid_idx))

    hide_idx = np.random.choice(
        valid_idx,
        size=hide_count,
        replace=False
    )

    for idx in hide_idx:
        hidden_positions.append((idx, col))

    val_masked.loc[hide_idx, col] = np.nan

print("Hidden values:", len(hidden_positions))

Hidden values: 424


In [13]:
import scipy
print(scipy.__version__)

1.17.1


In [14]:
val_spline = val_masked.copy()

val_spline[iv_cols] = (
    val_spline[iv_cols]
    .interpolate(
        method='spline',
        order=3,
        axis=1
    )
)

ValueError: Index column must be numeric or datetime type when using spline method other than linear. Try setting a numeric or datetime index column before interpolating.

In [15]:
val_spline = val_masked.copy()

val_spline[iv_cols] = (
    val_spline[iv_cols]
    .interpolate(
        method='linear',
        axis=1,
        limit_direction='both'
    )
)

print(val_spline[iv_cols].isna().sum().sum())

0


In [16]:
print(val_spline[iv_cols].isna().sum().sum())

0


In [17]:
from sklearn.metrics import mean_squared_error

actual_values = []
predicted_values = []

for idx, col in hidden_positions:

    actual = val_df.loc[idx, col]
    pred = val_spline.loc[idx, col]

    actual_values.append(actual)
    predicted_values.append(pred)

mse = mean_squared_error(
    actual_values,
    predicted_values
)

print("Baseline Interpolation MSE:", mse)

Baseline Interpolation MSE: 0.001003351190834619


In [18]:
import re

strike_map = {}

for col in iv_cols:
    strike = int(re.findall(r'(\d+)(?=CE|PE)', col)[0])
    strike_map[col] = strike

print(list(strike_map.items())[:5])

[('NIFTY27JAN2625200CE', 2625200), ('NIFTY27JAN2625300CE', 2625300), ('NIFTY27JAN2625400CE', 2625400), ('NIFTY27JAN2625500CE', 2625500), ('NIFTY27JAN2625600CE', 2625600)]


In [20]:
from scipy.interpolate import CubicSpline
import pandas as pd
import numpy as np

def spline_fill_row(row):

    row = row.copy()

    known_cols = [
        col for col in iv_cols
        if pd.notna(row[col])
    ]

    if len(known_cols) < 4:
        return row

    strikes = np.array([
        strike_map[col]
        for col in known_cols
    ])

    ivs = np.array([
        row[col]
        for col in known_cols
    ])

    spline = CubicSpline(
        strikes,
        ivs,
        extrapolate=True
    )

    missing_cols = [
        col for col in iv_cols
        if pd.isna(row[col])
    ]

    for col in missing_cols:
        row[col] = float(
            spline(strike_map[col])
        )

    return row

In [21]:
val_cubic = val_masked.copy()

val_cubic = val_cubic.apply(
    spline_fill_row,
    axis=1
)

print(
    val_cubic[iv_cols]
    .isna()
    .sum()
    .sum()
)

ValueError: `x` must be strictly increasing sequence.

In [22]:
print(type(val_masked.iloc[0]))

<class 'pandas.Series'>


In [23]:
ce_cols = [c for c in iv_cols if c.endswith('CE')]
pe_cols = [c for c in iv_cols if c.endswith('PE')]

print(len(ce_cols))
print(len(pe_cols))

14
14


In [24]:
from scipy.interpolate import CubicSpline
import numpy as np
import pandas as pd

ce_cols = [c for c in iv_cols if c.endswith('CE')]
pe_cols = [c for c in iv_cols if c.endswith('PE')]

def fill_group_with_spline(row, cols):

    known_cols = [
        col for col in cols
        if pd.notna(row[col])
    ]

    if len(known_cols) < 4:
        return row

    strikes = np.array([
        strike_map[col]
        for col in known_cols
    ])

    ivs = np.array([
        row[col]
        for col in known_cols
    ])

    order = np.argsort(strikes)

    strikes = strikes[order]
    ivs = ivs[order]

    spline = CubicSpline(
        strikes,
        ivs,
        extrapolate=True
    )

    missing_cols = [
        col for col in cols
        if pd.isna(row[col])
    ]

    for col in missing_cols:
        row[col] = float(
            spline(strike_map[col])
        )

    return row


def spline_fill_row(row):

    row = fill_group_with_spline(
        row,
        ce_cols
    )

    row = fill_group_with_spline(
        row,
        pe_cols
    )

    return row

In [25]:
val_cubic = val_masked.copy()

val_cubic = val_cubic.apply(
    spline_fill_row,
    axis=1
)

print(
    val_cubic[iv_cols]
    .isna()
    .sum()
    .sum()
)

0


In [26]:
from sklearn.metrics import mean_squared_error

actual = []
predicted = []

for idx, col in hidden_positions:

    actual.append(
        val_df.loc[idx, col]
    )

    predicted.append(
        val_cubic.loc[idx, col]
    )

mse = mean_squared_error(
    actual,
    predicted
)

print("CubicSpline MSE:", mse)

CubicSpline MSE: 0.0013713894463419705


In [27]:
import pandas as pd
import re

long_data = []

for _, row in train_df.iterrows():

    for col in iv_cols:

        iv = row[col]

        if pd.isna(iv):
            continue

        strike = int(col[-7:-2])

        option_type = col[-2:]

        long_data.append([
            row['datetime'],
            row['underlying_price'],
            strike,
            option_type,
            iv
        ])

long_df = pd.DataFrame(
    long_data,
    columns=[
        'datetime',
        'underlying_price',
        'strike',
        'option_type',
        'iv'
    ]
)

print(long_df.shape)
long_df.head()

(17455, 5)


,datetime,underlying_price,strike,option_type,iv
0,2026-01-07 09:15:00,26111.65,25200,CE,0.12662
1,2026-01-07 09:15:00,26111.65,25300,CE,0.12330
2,2026-01-07 09:15:00,26111.65,25400,CE,0.11741
3,2026-01-07 09:15:00,26111.65,25600,CE,0.11005
4,2026-01-07 09:15:00,26111.65,25700,CE,0.10576


In [28]:
print(long_df.shape)

print(long_df.head())

(17455, 5)
             datetime  underlying_price  strike option_type       iv
0 2026-01-07 09:15:00          26111.65   25200          CE  0.12662
1 2026-01-07 09:15:00          26111.65   25300          CE  0.12330
2 2026-01-07 09:15:00          26111.65   25400          CE  0.11741
3 2026-01-07 09:15:00          26111.65   25600          CE  0.11005
4 2026-01-07 09:15:00          26111.65   25700          CE  0.10576


In [29]:
long_df['moneyness'] = (
    long_df['strike']
    /
    long_df['underlying_price']
)

long_df['option_type'] = (
    long_df['option_type']
    .map({'CE':1, 'PE':0})
)

long_df['hour'] = long_df['datetime'].dt.hour

long_df['minute'] = long_df['datetime'].dt.minute

print(long_df.head())

             datetime  underlying_price  strike  option_type       iv  \
0 2026-01-07 09:15:00          26111.65   25200            1  0.12662   
1 2026-01-07 09:15:00          26111.65   25300            1  0.12330   
2 2026-01-07 09:15:00          26111.65   25400            1  0.11741   
3 2026-01-07 09:15:00          26111.65   25600            1  0.11005   
4 2026-01-07 09:15:00          26111.65   25700            1  0.10576   

   moneyness  hour  minute  
0   0.965086     9      15  
1   0.968916     9      15  
2   0.972746     9      15  
3   0.980405     9      15  
4   0.984235     9      15  


In [30]:
import lightgbm as lgb

print(lgb.__version__)

4.6.0


In [31]:
from sklearn.model_selection import train_test_split

features = [
    'underlying_price',
    'strike',
    'option_type',
    'moneyness',
    'hour',
    'minute'
]

X = long_df[features]
y = long_df['iv']

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print(X_train.shape)
print(X_valid.shape)

(13964, 6)
(3491, 6)


In [32]:
from lightgbm import LGBMRegressor

model = LGBMRegressor(
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42
)

model.fit(
    X_train,
    y_train
)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000229 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 561
[LightGBM] [Info] Number of data points in the train set: 13964, number of used features: 6
[LightGBM] [Info] Start training from score 0.133210


,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,learning_rate,0.05
,n_estimators,500
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [33]:
from sklearn.metrics import mean_squared_error

preds = model.predict(X_valid)

mse = mean_squared_error(
    y_valid,
    preds
)

print("LightGBM MSE:", mse)

LightGBM MSE: 2.3165428843733253e-05


In [34]:
long_df['distance_from_atm'] = (
    long_df['strike']
    - long_df['underlying_price']
)

long_df['abs_distance_from_atm'] = (
    long_df['distance_from_atm'].abs()
)

In [35]:
print(long_df.head())

             datetime  underlying_price  strike  option_type       iv  \
0 2026-01-07 09:15:00          26111.65   25200            1  0.12662   
1 2026-01-07 09:15:00          26111.65   25300            1  0.12330   
2 2026-01-07 09:15:00          26111.65   25400            1  0.11741   
3 2026-01-07 09:15:00          26111.65   25600            1  0.11005   
4 2026-01-07 09:15:00          26111.65   25700            1  0.10576   

   moneyness  hour  minute  distance_from_atm  abs_distance_from_atm  
0   0.965086     9      15            -911.65                 911.65  
1   0.968916     9      15            -811.65                 811.65  
2   0.972746     9      15            -711.65                 711.65  
3   0.980405     9      15            -511.65                 511.65  
4   0.984235     9      15            -411.65                 411.65  


In [36]:
features = [
    'underlying_price',
    'strike',
    'option_type',
    'moneyness',
    'hour',
    'minute',
    'distance_from_atm',
    'abs_distance_from_atm'
]

X = long_df[features]
y = long_df['iv']

print(X.shape)
print(y.shape)

(17455, 8)
(17455,)


In [37]:
from sklearn.model_selection import train_test_split

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print(X_train.shape)
print(X_valid.shape)

(13964, 8)
(3491, 8)


In [38]:
from lightgbm import LGBMRegressor

model2 = LGBMRegressor(
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42
)

model2.fit(
    X_train,
    y_train
)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000614 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1071
[LightGBM] [Info] Number of data points in the train set: 13964, number of used features: 8
[LightGBM] [Info] Start training from score 0.133210


,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,learning_rate,0.05
,n_estimators,500
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [39]:
from sklearn.metrics import mean_squared_error

preds2 = model2.predict(X_valid)

mse2 = mean_squared_error(
    y_valid,
    preds2
)

print("New MSE:", mse2)

New MSE: 2.4576657029553817e-05


In [40]:
old_mse = 2.3165428843733253e-05

print("Old MSE:", old_mse)
print("New MSE:", mse2)

if mse2 < old_mse:
    print("✅ Improved")
else:
    print("❌ Not Improved")

Old MSE: 2.3165428843733253e-05
New MSE: 2.4576657029553817e-05
❌ Not Improved


In [41]:
features = [
    'underlying_price',
    'strike',
    'option_type',
    'moneyness',
    'hour',
    'minute'
]

long_df = long_df.sort_values('datetime')

split_idx = int(0.8 * len(long_df))

train_long = long_df.iloc[:split_idx]
valid_long = long_df.iloc[split_idx:]

X_train = train_long[features]
y_train = train_long['iv']

X_valid = valid_long[features]
y_valid = valid_long['iv']

print(X_train.shape)
print(X_valid.shape)

(13964, 6)
(3491, 6)


In [42]:
from lightgbm import LGBMRegressor

chrono_model = LGBMRegressor(
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42
)

chrono_model.fit(
    X_train,
    y_train
)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000683 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 559
[LightGBM] [Info] Number of data points in the train set: 13964, number of used features: 6
[LightGBM] [Info] Start training from score 0.128550


,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,learning_rate,0.05
,n_estimators,500
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [43]:
from sklearn.metrics import mean_squared_error

chrono_preds = chrono_model.predict(X_valid)

chrono_mse = mean_squared_error(
    y_valid,
    chrono_preds
)

print("Chronological MSE:", chrono_mse)

Chronological MSE: 0.0003157791280735232


In [44]:
sample = pd.read_csv("sandbox_solution.csv")

print(sample.shape)
print(sample.head())

(5460, 2)
                                      id     value
0  07-01-2026 09:15||NIFTY27JAN2624100PE  0.143129
1  07-01-2026 09:15||NIFTY27JAN2625500CE  0.102447
2  07-01-2026 09:15||NIFTY27JAN2625800CE  0.102447
3  07-01-2026 09:20||NIFTY27JAN2624000PE  0.147267
4  07-01-2026 09:20||NIFTY27JAN2624200PE  0.147267


In [45]:
missing_count = df[iv_cols].isna().sum().sum()

print("Total Missing Values:", missing_count)

Total Missing Values: 5460


In [46]:
for col in iv_cols:
    if df[col].isna().sum() > 0:
        print(col, df[col].isna().sum())

NIFTY27JAN2625200CE 194
NIFTY27JAN2625300CE 184
NIFTY27JAN2625400CE 202
NIFTY27JAN2625500CE 181
NIFTY27JAN2625600CE 180
NIFTY27JAN2625700CE 201
NIFTY27JAN2625800CE 210
NIFTY27JAN2625900CE 182
NIFTY27JAN2626000CE 181
NIFTY27JAN2626100CE 183
NIFTY27JAN2626200CE 209
NIFTY27JAN2626300CE 187
NIFTY27JAN2626400CE 213
NIFTY27JAN2626500CE 184
NIFTY27JAN2623800PE 203
NIFTY27JAN2623900PE 201
NIFTY27JAN2624000PE 209
NIFTY27JAN2624100PE 176
NIFTY27JAN2624200PE 217
NIFTY27JAN2624300PE 189
NIFTY27JAN2624400PE 202
NIFTY27JAN2624500PE 195
NIFTY27JAN2624600PE 198
NIFTY27JAN2624700PE 175
NIFTY27JAN2624800PE 209
NIFTY27JAN2624900PE 195
NIFTY27JAN2625000PE 195
NIFTY27JAN2625100PE 205


In [47]:
missing_positions = []

for row_idx in df.index:
    for col in iv_cols:
        if pd.isna(df.loc[row_idx, col]):
            missing_positions.append((row_idx, col))

print(len(missing_positions))

5460


In [48]:
prediction_rows = []

for row_idx, col in missing_positions:

    strike = int(col[-7:-2])

    option_type = 1 if col.endswith("CE") else 0

    underlying_price = df.loc[row_idx, "underlying_price"]

    dt = df.loc[row_idx, "datetime"]

    prediction_rows.append({
        "row_idx": row_idx,
        "column": col,
        "datetime": dt,
        "underlying_price": underlying_price,
        "strike": strike,
        "option_type": option_type,
        "moneyness": strike / underlying_price,
        "hour": dt.hour,
        "minute": dt.minute
    })

pred_df = pd.DataFrame(prediction_rows)

print(pred_df.shape)
pred_df.head()

(5460, 9)


,row_idx,column,datetime,underlying_price,strike,option_type,moneyness,hour,minute
0,0,NIFTY27JAN2625500CE,2026-01-07 09:15:00,26111.65,25500,1,0.976576,9,15
1,0,NIFTY27JAN2625800CE,2026-01-07 09:15:00,26111.65,25800,1,0.988065,9,15
2,0,NIFTY27JAN2624100PE,2026-01-07 09:15:00,26111.65,24100,0,0.922960,9,15
3,1,NIFTY27JAN2625300CE,2026-01-07 09:20:00,26141.40,25300,1,0.967814,9,20
4,1,NIFTY27JAN2625400CE,2026-01-07 09:20:00,26141.40,25400,1,0.971639,9,20


In [49]:
X_pred = pred_df[
    [
        "underlying_price",
        "strike",
        "option_type",
        "moneyness",
        "hour",
        "minute"
    ]
]

print(X_pred.shape)

(5460, 6)


In [50]:
pred_df["predicted_iv"] = model.predict(X_pred)

pred_df.head()

,row_idx,column,datetime,underlying_price,strike,option_type,moneyness,hour,minute,predicted_iv
0,0,NIFTY27JAN2625500CE,2026-01-07 09:15:00,26111.65,25500,1,0.976576,9,15,0.112350
1,0,NIFTY27JAN2625800CE,2026-01-07 09:15:00,26111.65,25800,1,0.988065,9,15,0.102447
2,0,NIFTY27JAN2624100PE,2026-01-07 09:15:00,26111.65,24100,0,0.922960,9,15,0.165140
3,1,NIFTY27JAN2625300CE,2026-01-07 09:20:00,26141.40,25300,1,0.967814,9,20,0.098106
4,1,NIFTY27JAN2625400CE,2026-01-07 09:20:00,26141.40,25400,1,0.971639,9,20,0.100817


In [51]:
print(pred_df["predicted_iv"].describe())

count    5460.000000
mean        0.138028
std         0.033605
min         0.081971
25%         0.111809
50%         0.129758
75%         0.160512
max         0.281358
Name: predicted_iv, dtype: float64


In [52]:
sample = pd.read_csv("sandbox_solution.csv")

print(sample.head())
print(sample.shape)

                                      id     value
0  07-01-2026 09:15||NIFTY27JAN2624100PE  0.143129
1  07-01-2026 09:15||NIFTY27JAN2625500CE  0.102447
2  07-01-2026 09:15||NIFTY27JAN2625800CE  0.102447
3  07-01-2026 09:20||NIFTY27JAN2624000PE  0.147267
4  07-01-2026 09:20||NIFTY27JAN2624200PE  0.147267
(5460, 2)


In [53]:
submission = sample.copy()

submission["value"] = pred_df["predicted_iv"].values

submission.head()

,id,value
0,07-01-2026 09:15||NIFTY27JAN2624100PE,0.112350
1,07-01-2026 09:15||NIFTY27JAN2625500CE,0.102447
2,07-01-2026 09:15||NIFTY27JAN2625800CE,0.165140
3,07-01-2026 09:20||NIFTY27JAN2624000PE,0.098106
4,07-01-2026 09:20||NIFTY27JAN2624200PE,0.100817


In [54]:
submission.to_csv(
    "final_submission.csv",
    index=False
)

print("Submission saved!")

Submission saved!


In [55]:
print(submission.shape)
print(submission.head())

(5460, 2)
                                      id     value
0  07-01-2026 09:15||NIFTY27JAN2624100PE  0.112350
1  07-01-2026 09:15||NIFTY27JAN2625500CE  0.102447
2  07-01-2026 09:15||NIFTY27JAN2625800CE  0.165140
3  07-01-2026 09:20||NIFTY27JAN2624000PE  0.098106
4  07-01-2026 09:20||NIFTY27JAN2624200PE  0.100817


In [56]:
check = pd.read_csv("final_submission.csv")
print(check.head())
print(check.shape)

                                      id     value
0  07-01-2026 09:15||NIFTY27JAN2624100PE  0.112350
1  07-01-2026 09:15||NIFTY27JAN2625500CE  0.102447
2  07-01-2026 09:15||NIFTY27JAN2625800CE  0.165140
3  07-01-2026 09:20||NIFTY27JAN2624000PE  0.098106
4  07-01-2026 09:20||NIFTY27JAN2624200PE  0.100817
(5460, 2)


In [57]:
filled_df = df.copy()

for _, row in pred_df.iterrows():
    filled_df.loc[
        row["row_idx"],
        row["column"]
    ] = row["predicted_iv"]

print(
    filled_df[iv_cols]
    .isna()
    .sum()
    .sum()
)

0


In [58]:
filled_df.to_csv(
    "filled_dataset.csv",
    index=False
)

print("Filled dataset saved!")

Filled dataset saved!


In [59]:
import pandas as pd

ORIGINAL_DATASET_PATH = "dataset.csv"

SEPARATOR = "||"

def generate_solution(
    filled_path: str,
    output_path: str = "submission.csv"
):
    original = pd.read_csv(ORIGINAL_DATASET_PATH)
    filled   = pd.read_csv(filled_path)

    feature_cols = [
        c for c in original.columns
        if c != "datetime"
    ]

    rows = []

    for col in feature_cols:
        was_missing = original[col].isna()

        for idx in original.index[was_missing]:

            dt = original.loc[idx, "datetime"]

            uid = f"{dt}{SEPARATOR}{col}"

            val = filled.loc[idx, col]

            rows.append({
                "id": uid,
                "value": val
            })

    solution = pd.DataFrame(
        rows,
        columns=["id", "value"]
    )

    solution = (
        solution
        .sort_values("id")
        .reset_index(drop=True)
    )

    solution.to_csv(
        output_path,
        index=False
    )

    print(
        f"Solution saved -> {output_path} ({len(solution)} rows)"
    )

generate_solution(
    "filled_dataset.csv",
    "submission.csv"
)

Solution saved -> submission.csv (5460 rows)


In [60]:
sub = pd.read_csv("submission.csv")

print(sub.shape)
print(sub.head())

(5460, 2)
                                      id     value
0  07-01-2026 09:15||NIFTY27JAN2624100PE  0.165140
1  07-01-2026 09:15||NIFTY27JAN2625500CE  0.112350
2  07-01-2026 09:15||NIFTY27JAN2625800CE  0.102447
3  07-01-2026 09:20||NIFTY27JAN2624000PE  0.171364
4  07-01-2026 09:20||NIFTY27JAN2624200PE  0.160089


In [61]:
print(sub["value"].isna().sum())

0


In [62]:
surface_df = df.copy()

surface_df[iv_cols] = (
    surface_df[iv_cols]
    .interpolate(
        method="linear",
        axis=1,
        limit_direction="both"
    )
)

print(surface_df[iv_cols].isna().sum().sum())

0


In [63]:
surface_long = surface_df.melt(
    id_vars=["datetime", "underlying_price"],
    value_vars=iv_cols,
    var_name="contract",
    value_name="spline_pred"
)

print(surface_long.shape)
surface_long.head()

(27300, 4)


,datetime,underlying_price,contract,spline_pred
0,2026-01-07 09:15:00,26111.65,NIFTY27JAN2625200CE,0.12662
1,2026-01-07 09:20:00,26141.40,NIFTY27JAN2625200CE,0.08632
2,2026-01-07 09:25:00,26139.35,NIFTY27JAN2625200CE,0.09147
3,2026-01-07 09:30:00,26128.95,NIFTY27JAN2625200CE,0.10860
4,2026-01-07 09:35:00,26131.90,NIFTY27JAN2625200CE,0.10462


In [64]:
surface_long["strike"] = (
    surface_long["contract"]
    .str.extract(r'(\d+)(CE|PE)')[0]
    .astype(int)
)

surface_long["option_type"] = (
    surface_long["contract"]
    .str.extract(r'(CE|PE)')[0]
)

surface_long["option_type"] = (
    surface_long["option_type"]
    .map({"CE":1, "PE":0})
)

surface_long.head()

,datetime,underlying_price,contract,spline_pred,strike,option_type
0,2026-01-07 09:15:00,26111.65,NIFTY27JAN2625200CE,0.12662,2625200,1
1,2026-01-07 09:20:00,26141.40,NIFTY27JAN2625200CE,0.08632,2625200,1
2,2026-01-07 09:25:00,26139.35,NIFTY27JAN2625200CE,0.09147,2625200,1
3,2026-01-07 09:30:00,26128.95,NIFTY27JAN2625200CE,0.10860,2625200,1
4,2026-01-07 09:35:00,26131.90,NIFTY27JAN2625200CE,0.10462,2625200,1


In [65]:
long_df = long_df.merge(
    surface_long[
        [
            "datetime",
            "strike",
            "option_type",
            "spline_pred"
        ]
    ],
    on=[
        "datetime",
        "strike",
        "option_type"
    ],
    how="left"
)

print(long_df.shape)
print(long_df["spline_pred"].isna().sum())

(17455, 11)
17455


In [66]:
print(long_df.dtypes)

print("\n-----------------\n")

print(surface_long.dtypes)

datetime                 datetime64[us]
underlying_price                float64
strike                            int64
option_type                       int64
iv                              float64
moneyness                       float64
hour                              int32
minute                            int32
distance_from_atm               float64
abs_distance_from_atm           float64
spline_pred                     float64
dtype: object

-----------------

datetime            datetime64[us]
underlying_price           float64
contract                       str
spline_pred                float64
strike                       int64
option_type                  int64
dtype: object


In [67]:
print(long_df[["datetime","strike","option_type"]].head())

print("\n-----------------\n")

print(surface_long[["datetime","strike","option_type"]].head())

             datetime  strike  option_type
0 2026-01-07 09:15:00   25200            1
1 2026-01-07 09:15:00   25100            0
2 2026-01-07 09:15:00   25000            0
3 2026-01-07 09:15:00   24900            0
4 2026-01-07 09:15:00   24800            0

-----------------

             datetime   strike  option_type
0 2026-01-07 09:15:00  2625200            1
1 2026-01-07 09:20:00  2625200            1
2 2026-01-07 09:25:00  2625200            1
3 2026-01-07 09:30:00  2625200            1
4 2026-01-07 09:35:00  2625200            1


In [68]:
surface_long["contract"].head(20)

0     NIFTY27JAN2625200CE
1     NIFTY27JAN2625200CE
2     NIFTY27JAN2625200CE
3     NIFTY27JAN2625200CE
4     NIFTY27JAN2625200CE
5     NIFTY27JAN2625200CE
6     NIFTY27JAN2625200CE
7     NIFTY27JAN2625200CE
8     NIFTY27JAN2625200CE
9     NIFTY27JAN2625200CE
10    NIFTY27JAN2625200CE
11    NIFTY27JAN2625200CE
12    NIFTY27JAN2625200CE
13    NIFTY27JAN2625200CE
14    NIFTY27JAN2625200CE
15    NIFTY27JAN2625200CE
16    NIFTY27JAN2625200CE
17    NIFTY27JAN2625200CE
18    NIFTY27JAN2625200CE
19    NIFTY27JAN2625200CE
Name: contract, dtype: str

In [69]:
surface_long["strike"] = (
    surface_long["contract"]
    .str.extract(r'NIFTY27JAN26(\d+)(CE|PE)')[0]
    .astype(int)
)

surface_long["option_type"] = (
    surface_long["contract"]
    .str.extract(r'(CE|PE)')[0]
    .map({"CE": 1, "PE": 0})
)

surface_long[["contract","strike","option_type"]].head()

,contract,strike,option_type
0,NIFTY27JAN2625200CE,25200,1
1,NIFTY27JAN2625200CE,25200,1
2,NIFTY27JAN2625200CE,25200,1
3,NIFTY27JAN2625200CE,25200,1
4,NIFTY27JAN2625200CE,25200,1


In [70]:
long_df = long_df.drop(columns=["spline_pred"], errors="ignore")

long_df = long_df.merge(
    surface_long[
        [
            "datetime",
            "strike",
            "option_type",
            "spline_pred"
        ]
    ],
    on=[
        "datetime",
        "strike",
        "option_type"
    ],
    how="left"
)

print(long_df.shape)
print(long_df["spline_pred"].isna().sum())

(17455, 11)
0


In [71]:
print(long_df["spline_pred"].isna().sum())

0


In [72]:
features = [
    "underlying_price",
    "strike",
    "option_type",
    "moneyness",
    "hour",
    "minute",
    "distance_from_atm",
    "abs_distance_from_atm",
    "spline_pred"
]

X = long_df[features]
y = long_df["iv"]

print(X.shape)
print(y.shape)
X.head()

(17455, 9)
(17455,)


,underlying_price,strike,option_type,moneyness,hour,minute,distance_from_atm,abs_distance_from_atm,spline_pred
0,26111.65,25200,1,0.965086,9,15,-911.65,911.65,0.12662
1,26111.65,25100,0,0.961257,9,15,-1011.65,1011.65,0.11150
2,26111.65,25000,0,0.957427,9,15,-1111.65,1111.65,0.11631
3,26111.65,24900,0,0.953597,9,15,-1211.65,1211.65,0.12142
4,26111.65,24800,0,0.949768,9,15,-1311.65,1311.65,0.12640


In [73]:
from sklearn.model_selection import train_test_split

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Train:", X_train.shape)
print("Valid:", X_valid.shape)

Train: (13964, 9)
Valid: (3491, 9)


In [74]:
from lightgbm import LGBMRegressor

model = LGBMRegressor(
    n_estimators=1000,
    learning_rate=0.03,
    num_leaves=63,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

model.fit(
    X_train,
    y_train
)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000328 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1326
[LightGBM] [Info] Number of data points in the train set: 13964, number of used features: 9
[LightGBM] [Info] Start training from score 0.133256


,boosting_type,'gbdt'
,num_leaves,63
,max_depth,-1
,learning_rate,0.03
,n_estimators,1000
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [75]:
from sklearn.metrics import mean_squared_error

preds = model.predict(X_valid)

mse = mean_squared_error(
    y_valid,
    preds
)

print("NEW MSE =", mse)

NEW MSE = 4.28826183126901e-07


In [76]:
X_pred = pred_df[
    [
        "underlying_price",
        "strike",
        "option_type",
        "moneyness",
        "hour",
        "minute",
        "distance_from_atm",
        "abs_distance_from_atm",
        "spline_pred"
    ]
]

print(X_pred.shape)

KeyError: "['distance_from_atm', 'abs_distance_from_atm', 'spline_pred'] not in index"

In [77]:
print(pred_df.columns.tolist())

['row_idx', 'column', 'datetime', 'underlying_price', 'strike', 'option_type', 'moneyness', 'hour', 'minute', 'predicted_iv']


In [78]:
pred_df["distance_from_atm"] = (
    pred_df["strike"] - pred_df["underlying_price"]
)

pred_df["abs_distance_from_atm"] = (
    pred_df["distance_from_atm"].abs()
)

In [79]:
print(surface_long.columns.tolist())

['datetime', 'underlying_price', 'contract', 'spline_pred', 'strike', 'option_type']


In [80]:
pred_df = pred_df.merge(
    surface_long[
        [
            "datetime",
            "strike",
            "option_type",
            "spline_pred"
        ]
    ],
    on=[
        "datetime",
        "strike",
        "option_type"
    ],
    how="left"
)

In [81]:
print(pred_df.columns.tolist())
print(pred_df["spline_pred"].isna().sum())

['row_idx', 'column', 'datetime', 'underlying_price', 'strike', 'option_type', 'moneyness', 'hour', 'minute', 'predicted_iv', 'distance_from_atm', 'abs_distance_from_atm', 'spline_pred']
0


In [82]:
X_pred = pred_df[
    [
        "underlying_price",
        "strike",
        "option_type",
        "moneyness",
        "hour",
        "minute",
        "distance_from_atm",
        "abs_distance_from_atm",
        "spline_pred"
    ]
]

print(X_pred.shape)

(5460, 9)


In [83]:
pred_df["predicted_iv"] = model.predict(X_pred)

print(pred_df["predicted_iv"].describe())

count    5460.000000
mean        0.142272
std         0.037347
min         0.075587
25%         0.113300
50%         0.132801
75%         0.164219
max         0.279808
Name: predicted_iv, dtype: float64


In [84]:
filled_df = df.copy()

for _, row in pred_df.iterrows():
    filled_df.loc[
        row["row_idx"],
        row["column"]
    ] = row["predicted_iv"]

print(
    "Remaining missing:",
    filled_df[iv_cols].isna().sum().sum()
)

Remaining missing: 0


In [85]:
filled_df.to_csv(
    "filled_dataset_v2.csv",
    index=False
)

print("saved")

saved


In [86]:
generate_solution(
    "filled_dataset_v2.csv",
    "submission_v2.csv"
)

Solution saved -> submission_v2.csv (5460 rows)


In [87]:
sub = pd.read_csv("submission_v2.csv")

print(sub.shape)
print(sub.head())

(5460, 2)
                                      id     value
0  07-01-2026 09:15||NIFTY27JAN2624100PE  0.163422
1  07-01-2026 09:15||NIFTY27JAN2625500CE  0.114033
2  07-01-2026 09:15||NIFTY27JAN2625800CE  0.101267
3  07-01-2026 09:20||NIFTY27JAN2624000PE  0.170385
4  07-01-2026 09:20||NIFTY27JAN2624200PE  0.160059


In [88]:
import pickle

with open("lgbm_v1.pkl", "wb") as f:
    pickle.dump(model, f)

print("Model saved")

Model saved


In [1]:
1+1

2